# 非线性规划（NLP）：一个模型一个 Notebook

## 这个模型解决什么问题
目标或约束里出现平方、乘积、指数、开方、三角函数时，线性规划就不够用了。

```
min  f(x)
s.t. g_i(x) <= 0     不等式约束
     h_j(x) = 0      等式约束
     lb <= x <= ub
```

典型场景：定价（价格越高销量越低，收益是二次的）、库存（持有成本随库存非线性增长）、
选址（距离是欧氏距离，天然带根号）、参数拟合。

## 为什么强调「多起点」
非线性问题通常**有多个局部最优**。SLSQP 从哪个初值出发，就可能停在哪个局部最优。
所以比赛里至少要做两件事：
1. 多起点（随机撒十几个初值，取最好的）；
2. 报告"目标值对初值的敏感性"——这是评委很爱问的点。

## 目录
0 环境 / 1 配置 / 2 工具 / 3 求解（含多起点）/ 4 示例 A（带约束）/ 5 示例 B（定价问题）/ 6 导出

> 依赖 `numpy pandas scipy`。


In [1]:
# -*- coding: utf-8 -*-
# =====================================================================
# 0. 环境与路径（每个 Notebook 都带这一段，保证单独打开就能跑）
# =====================================================================
# 依赖：numpy、pandas、scipy（>=1.9，用到 milp）
# 可选：matplotlib（画图）、openpyxl（导出 xlsx）
# 缺哪个就装：pip install numpy pandas scipy matplotlib openpyxl
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize as sp_opt
from scipy.optimize import Bounds, LinearConstraint

try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 不设中文字体，图里的标题和标签会变成方块
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


def find_root(marker="导入数据"):
    """向上逐级找含 导入数据/ 的目录。

    Notebook 放在 优化类模型/ 下也能找到项目根，换电脑、换目录都不用改死路径。
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return Path.cwd()


ROOT = find_root()
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("项目根目录：", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp_opt.__name__ and __import__("scipy").__version__)


项目根目录： D:\git项目
numpy 2.4.6 | pandas 3.1.0 | scipy 1.17.1


In [2]:
# =====================================================================
# 1. 配置区
# =====================================================================
CFG = dict(
    method="SLSQP",      # 有约束用 SLSQP；无约束可换 BFGS / Nelder-Mead
    multi_starts=8,      # 多起点次数（>=8 才比较稳）
    seed=42,             # 随机种子，保证可复现
    maximize=False,      # True 时内部对目标取负
    tag="非线性规划",
    make_plots=True,
    save_excel=True,
)
print("配置：", CFG)


配置： {'method': 'SLSQP', 'multi_starts': 8, 'seed': 42, 'maximize': False, 'tag': '非线性规划', 'make_plots': True, 'save_excel': True}


## 2. 工具

非线性规划通常不需要读表，目标函数直接写在代码里；如果参数来自附件，用下面的 `load_table` 读进来即可。


In [3]:
# =====================================================================
# 2. 数据接口：读文件 → 清洗 → 变成模型能吃的表
# =====================================================================
# 这一格是「原始数据直接放进来」的关键：不用你手动去空格、转数字、去重。
# 如果数据已经处理好，可以跳过清洗，直接用后面的 load_matrix 读矩阵。
#
# ⚠️ 999 是否算缺失取决于题目：若它是真实观测值，把它从 NA_MARKERS 里删掉。
NA_MARKERS = ("", "NA", "N/A", "na", "缺测", "未填", "无", "999", 999, "9999", 9999)


def load_table(data, sheet_name=0, encoding="utf-8-sig"):
    """读 csv / xlsx，也接受现成的 DataFrame。

    - csv 先按 utf-8-sig 读（吃掉 Excel 导出的 BOM），失败再退回 gbk
    - 列名首尾空格自动去掉（" 单耗" → "单耗"）
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        p = Path(data)
        if not p.exists():
            raise FileNotFoundError(f"找不到数据文件：{p}")
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p, sheet_name=sheet_name)
        else:
            try:
                df = pd.read_csv(p, encoding=encoding)
            except UnicodeDecodeError:
                df = pd.read_csv(p, encoding="gbk")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def clean_frame(df, na_markers=NA_MARKERS, drop_missing_rate=50.0,
                fill_num="median", diagnostics=None):
    """把原始附件洗成能建模的表，并把每一步记进 diagnostics。

    顺序有因果，别随意调换：
    ① 哨兵值 → NaN       999 / 缺测 / 未填 统一当缺失
    ② 文本列去空格       否则 "东部" 与 " 东部" 会被当成两个类别
    ③ 文本数字转数值     " 12.5 " 这种带空格的数字
    ④ 删完全重复行
    ⑤ 高缺失列删列       删列不删行，保住样本量
    ⑥ 缺失填补           数值列中位数/均值，文本列众数

    参数
    ----
    diagnostics : 传一个 list 进来，会往里追加每一步的说明，最后可以导出给论文用
    """
    diag = diagnostics if diagnostics is not None else []
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]

    before = int(d.isna().sum().sum())
    d = d.replace(list(na_markers), np.nan)
    diag.append({"步骤": "哨兵值转缺失",
                 "说明": f"新增缺失 {int(d.isna().sum().sum()) - before} 个"})

    obj_cols = [c for c in d.columns
                if pd.api.types.is_string_dtype(d[c]) or pd.api.types.is_object_dtype(d[c])]
    for c in obj_cols:
        d[c] = d[c].astype("string").str.strip()
    if obj_cols:
        diag.append({"步骤": "文本列去空格", "说明": f"处理 {len(obj_cols)} 列"})

    num_cols, cat_cols = [], []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(c)
            continue
        conv = pd.to_numeric(s.astype(str).str.strip(), errors="coerce")
        if conv.notna().sum() / max(s.notna().sum(), 1) > 0.9:
            d[c] = conv
            num_cols.append(c)
        else:
            cat_cols.append(c)
    diag.append({"步骤": "列类型识别",
                 "说明": f"数值列 {len(num_cols)} 个、类别列 {len(cat_cols)} 个"})

    n_dup = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first").reset_index(drop=True)
    diag.append({"步骤": "去完全重复", "说明": f"删除 {n_dup} 行"})

    rate = d.isna().mean() * 100
    drop = list(rate[rate >= drop_missing_rate].index)
    if drop:
        d = d.drop(columns=drop)
        diag.append({"步骤": "删除高缺失列",
                     "说明": f"删除 {drop}（缺失率 ≥ {drop_missing_rate}%），删列不删行"})

    filled = []
    for c in num_cols:
        if c in d.columns and d[c].isna().any():
            fill = {"median": d[c].median(), "mean": d[c].mean(), "zero": 0}[fill_num]
            d[c] = d[c].fillna(fill)
            filled.append(c)
    for c in cat_cols:
        if c in d.columns and d[c].isna().any():
            m = d[c].mode(dropna=True)
            d[c] = d[c].fillna(m.iat[0] if len(m) else "未知")
            filled.append(c)
    if filled:
        diag.append({"步骤": "缺失填补", "说明": f"填补 {filled}，数值方式 {fill_num}"})

    diag.append({"步骤": "清洗完成", "说明": f"{d.shape[0]} 行 × {d.shape[1]} 列"})
    return d


def pivot_long(df, index, columns, values, agg="sum"):
    """长表转宽表：产品-资源-单耗 三列 → 产品 × 资源 矩阵。

    真实附件常见这种长表；同一 (产品, 资源) 出现多次时按 agg 聚合。
    """
    wide = df.pivot_table(index=index, columns=columns, values=values, aggfunc=agg).reset_index()
    wide.columns = [str(c) for c in wide.columns]
    return wide


def load_matrix(data, sheet_name=0):
    """读成本矩阵 / 距离矩阵：第一列当行名，其余列转成数值。"""
    df = load_table(data, sheet_name)
    if df.shape[1] > 1 and not pd.api.types.is_numeric_dtype(df.iloc[:, 0]):
        df = df.set_index(df.columns[0])
    df.index = [str(i) for i in df.index]
    df.columns = [str(c) for c in df.columns]
    return df.astype(float)


## 3. 求解：多起点 + SLSQP

单起点很容易停在局部最优，这一节的重点是**多起点**和**约束核对**。


In [4]:
# =====================================================================
# 3. 求解：多起点 + SLSQP
# =====================================================================
def solve_nlp(fun, x0, bounds=None, constraints=(), method="SLSQP",
              maximize=False, multi_starts=1, seed=0, maxiter=1000):
    """非线性规划求解，支持多起点。

    fun         : 目标函数，输入一维数组返回标量
    x0          : 初值
    bounds      : [(lb, ub), ...]，lb/ub 可为 None
    constraints : scipy 约束字典列表，例如
                  {"type": "ineq", "fun": lambda x: 10 - x[0] - x[1]}   表示 >= 0
                  {"type": "eq",   "fun": lambda x: x[0] + x[1] - 5}    表示 = 0
    multi_starts: 随机初值个数；>=8 能明显降低陷入局部最优的概率
    """
    sign = -1.0 if maximize else 1.0
    f = lambda x: sign * float(fun(x))
    x0 = np.asarray(x0, dtype=float).ravel()
    n = len(x0)
    bnds = list(bounds) if bounds is not None else [(None, None)] * n

    # 生成起点：第一个用用户给的 x0，其余在边界内均匀撒点
    rng = np.random.default_rng(seed)
    lo = np.array([b[0] if b[0] is not None else -5.0 for b in bnds])
    hi = np.array([b[1] if b[1] is not None else 5.0 for b in bnds])
    starts = [x0] + [rng.uniform(lo, hi) for _ in range(max(multi_starts - 1, 0))]

    trials = []
    best = None
    for k, s in enumerate(starts):
        r = sp_opt.minimize(f, s, method=method, bounds=bnds,
                            constraints=list(constraints),
                            options={"maxiter": maxiter})
        trials.append({"起点": k + 1, "初值": np.round(s, 3).tolist(),
                       "目标值": round(sign * r.fun, 6),
                       "是否收敛": bool(r.success)})
        if best is None or r.fun < best.fun:
            best = r

    return dict(x=best.x, objective=sign * float(best.fun), ok=bool(best.success),
                message=str(best.message), method=method,
                trials=pd.DataFrame(trials), n_starts=len(starts))


def check_constraints(constraints, x, tol=1e-6):
    """核对约束是否满足（论文里最好附一张这样的表）。"""
    rows = []
    for k, c in enumerate(constraints, start=1):
        v = float(c["fun"](x))
        kind = "≥0" if c["type"] == "ineq" else "=0"
        ok = v >= -tol if c["type"] == "ineq" else abs(v) <= 1e-4
        rows.append({"约束": f"约束{k}", "类型": kind, "当前值": round(v, 8),
                     "是否满足": "✔" if ok else "✘"})
    return pd.DataFrame(rows)


## 4. 示例 A：带约束的二次目标


In [5]:
# =====================================================================
# 4. 示例 A：带约束的二次目标
# =====================================================================
# min (x1-2)^2 + (x2-3)^2 + 1
# s.t. x1 + x2 <= 10, 0 <= xi <= 10
# 无约束最优是 (2, 3)，目标值 1；它满足约束，所以约束不起作用。
fun_a = lambda x: (x[0] - 2) ** 2 + (x[1] - 3) ** 2 + 1
cons_a = ({"type": "ineq", "fun": lambda x: 10 - x[0] - x[1]},)

res_a = solve_nlp(fun_a, x0=[0, 0], bounds=[(0, 10), (0, 10)],
                  constraints=cons_a, method=CFG["method"],
                  multi_starts=CFG["multi_starts"], seed=CFG["seed"])
print("最优解 x =", np.round(res_a["x"], 4), "| 目标值 =", round(res_a["objective"], 6))
print("\n多起点对比：")
print(res_a["trials"].to_string(index=False))
print("\n约束核对：")
print(check_constraints(cons_a, res_a["x"]).to_string(index=False))
print("\n理论最优 (2, 3)、目标值 1 —— 对上说明求解正确。")


最优解 x = [2. 3.] | 目标值 = 1.0

多起点对比：
 起点             初值  目标值  是否收敛
  1     [0.0, 0.0]  1.0  True
  2  [7.74, 4.389]  1.0  True
  3 [8.586, 6.974]  1.0  True
  4 [0.942, 9.756]  1.0  True
  5 [7.611, 7.861]  1.0  True
  6 [1.281, 4.504]  1.0  True
  7 [3.708, 9.268]  1.0  True
  8 [6.439, 8.228]  1.0  True

约束核对：
 约束 类型  当前值 是否满足
约束1 ≥0  5.0    ✔

理论最优 (2, 3)、目标值 1 —— 对上说明求解正确。


## 5. 示例 B：定价问题（与解析解对照）


In [6]:
# =====================================================================
# 5. 示例 B：定价问题（收益是二次的，最大化）
# =====================================================================
# 三个产品，价格 p_i，需求 d_i = a_i - b_i * p_i（价格越高卖得越少），
# 单位成本 c_i，总利润 = Σ (p_i - c_i) * d_i。
# 这是标准二次规划，但也能用 SLSQP 解；注意价格下限不能低于成本。
a = np.array([200.0, 160.0, 240.0])     # 价格为零时的潜在需求
b = np.array([2.5, 2.0, 3.0])           # 需求对价格的敏感度
c_cost = np.array([20.0, 18.0, 25.0])   # 单位成本

def profit(p):
    demand = np.maximum(a - b * p, 0)   # 需求不能为负
    return float(((p - c_cost) * demand).sum())

res_b = solve_nlp(lambda p: -profit(p),          # 最大化 → 对目标取负
                  x0=[40, 40, 40],
                  bounds=[(c_cost[i], a[i] / b[i]) for i in range(3)],
                  maximize=False, multi_starts=CFG["multi_starts"],
                  seed=CFG["seed"])
optimal_p = res_b["x"]
print("最优定价：", dict(zip(["产品1", "产品2", "产品3"], np.round(optimal_p, 2))))
print("最大利润：", round(profit(optimal_p), 2))

# 解析解对照：单产品利润 (p-c)(a-bp) 对 p 求导 → p* = (a + b*c) / (2b)
p_star = (a + b * c_cost) / (2 * b)
print("\n解析最优价：", np.round(p_star, 2), "| 最大利润：", round(profit(p_star), 2))
print("数值解与解析解差异：", np.round(np.abs(optimal_p - p_star), 6))

print("\n不同定价方案对比：")
compare = pd.DataFrame({
    "方案": ["成本定价", "初始猜测", "数值最优", "解析最优"],
    "价格": [np.round(c_cost, 2), [40, 40, 40],
             np.round(optimal_p, 2), np.round(p_star, 2)],
})
compare["总利润"] = [round(profit(np.array(v)), 2) for v in compare["价格"]]
print(compare.to_string(index=False))


最优定价： {'产品1': np.float64(50.0), '产品2': np.float64(49.0), '产品3': np.float64(52.5)}
最大利润： 6440.75

解析最优价： [50.  49.  52.5] | 最大利润： 6440.75
数值解与解析解差异： [2.e-06 0.e+00 2.e-06]

不同定价方案对比：
  方案                 价格     总利润
成本定价 [20.0, 18.0, 25.0]    0.00
初始猜测       [40, 40, 40] 5560.00
数值最优 [50.0, 49.0, 52.5] 6440.75
解析最优 [50.0, 49.0, 52.5] 6440.75


## 6. 结果导出


In [7]:
# =====================================================================
# 结果导出：CSV + 多 Sheet Excel + 图
# =====================================================================
def export_result(tables, cfg, fig=None, fig_name="结果图"):
    """把结果写成文件，返回写出的路径。

    tables : {Sheet 名: DataFrame}，第一张建议放"摘要"
    cfg    : 配置字典，用 tag / make_plots / save_excel 三个键
    fig    : 传了 matplotlib 图对象就保存；cfg["make_plots"]=False 时连图都不生成
    """
    d = OUT_DIR / f"优化_{cfg['tag']}"
    d.mkdir(parents=True, exist_ok=True)
    written = {}

    first = next(iter(tables.values()))
    p_csv = d / f"{cfg['tag']}_主表.csv"
    first.to_csv(p_csv, index=False, encoding="utf-8-sig")
    written["主表"] = p_csv

    if cfg.get("save_excel", True):
        p_xlsx = d / f"{cfg['tag']}_结果汇总.xlsx"
        with pd.ExcelWriter(p_xlsx, engine="openpyxl") as w:
            for name, frame in tables.items():
                frame.to_excel(w, sheet_name=str(name)[:28], index=False)
        written["汇总"] = p_xlsx
        print(f"[表格] {p_xlsx.name}（{len(tables)} 个 Sheet）")

    if fig is not None and cfg.get("make_plots", True) and HAS_MPL:
        p_png = d / f"{cfg['tag']}_{fig_name}.png"
        fig.savefig(p_png, dpi=150, bbox_inches="tight")
        if "agg" not in matplotlib.get_backend().lower():   # 无界面后端不弹窗，也不报警告
            plt.show()
        plt.close(fig)          # 关掉画布：反复运行不会在内存里堆图
        written["图片"] = p_png
        print(f"[图片] {p_png.name}")

    print("[输出] ", d)
    return written


In [8]:
# =====================================================================
# 6. 结果导出与灵敏度图
# =====================================================================
fig = None
if CFG["make_plots"] and HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    # 左图：目标函数在 x1-x2 平面上的等高线 + 最优点
    g = np.linspace(0, 10, 120)
    X, Y = np.meshgrid(g, g)
    Z = (X - 2) ** 2 + (Y - 3) ** 2 + 1
    cs = axes[0].contourf(X, Y, Z, levels=20, cmap="Blues")
    axes[0].plot(*res_a["x"], "r*", ms=14, label="最优点")
    axes[0].set_xlabel("x1"); axes[0].set_ylabel("x2")
    axes[0].set_title("目标函数等高线与最优点")
    axes[0].legend()
    fig.colorbar(cs, ax=axes[0], shrink=.85)

    # 右图：价格从成本到上限扫描时，利润怎么变
    grid = np.linspace(c_cost[0], a[0] / b[0], 100)
    profits = []
    for p1 in grid:
        p = optimal_p.copy(); p[0] = p1
        profits.append(profit(p))
    axes[1].plot(grid, profits, color="#4a9d6a")
    axes[1].axvline(optimal_p[0], ls="--", c="#d95f5f",
                    label=f"最优价 {optimal_p[0]:.1f}")
    axes[1].set_xlabel("产品1 定价"); axes[1].set_ylabel("总利润")
    axes[1].set_title("单产品价格灵敏度")
    axes[1].legend(fontsize=9)
    fig.tight_layout()

written = export_result(
    {"结果摘要": pd.DataFrame([{"最优目标值": round(res_b["objective"], 4),
                              "起点数": res_b["n_starts"],
                              "方法": res_b["method"],
                              "收敛": res_b["ok"]}]),
     "最优定价": pd.DataFrame({"产品": ["产品1", "产品2", "产品3"],
                            "最优价格": np.round(optimal_p, 4),
                            "解析解": np.round(p_star, 4),
                            "单位成本": c_cost}),
     "多起点对比": res_a["trials"],
     "约束核对": check_constraints(cons_a, res_a["x"])},
    CFG, fig=fig, fig_name="非线性结果")
print(written)


[表格] 非线性规划_结果汇总.xlsx（4 个 Sheet）
[图片] 非线性规划_非线性结果.png
[输出]  D:\git项目\输出数据\优化_非线性规划
{'主表': WindowsPath('D:/git项目/输出数据/优化_非线性规划/非线性规划_主表.csv'), '汇总': WindowsPath('D:/git项目/输出数据/优化_非线性规划/非线性规划_结果汇总.xlsx'), '图片': WindowsPath('D:/git项目/输出数据/优化_非线性规划/非线性规划_非线性结果.png')}


换题目时按三步走：

1. 数据放进 `导入数据/`（csv / xlsx 都行）；
2. 改本 Notebook 第 1 个代码单元里的 `CFG`：文件名、列名映射、方向、输出名；
3. `Run All`，结果写到 `输出数据/优化_{tag}/`，同名覆盖，不会越跑越多。

**两种输入方式怎么选**

- 数据还脏（有空行、重复、999、文本数字、列名带空格）→ 走「原始数据」示例，让 `clean_frame` 处理；
- 数据已经清洗/标准化完（比如上一节 `数据学习.ipynb` 输出的 `final_cleaned.csv`）→ 走「已处理数据」示例，直接给矩阵或宽表。

不要混：**同一次建模只用一种入口**，否则等于把同一份数据预处理了两遍。
